# DIY Copilot
## Lesson: Prompt Engineering

This lesson is intended to improve your prompt engineering skills. The prompts of commercial coding assistants are long, carefully engineered documents – and they use exactly the patterns you will practise here.

During this lesson you will learn how to ...
* use a system prompt to define model behaviour
* extend the system prompt to create workflows
* specify the output format, so that the answer can be processed by code

Our scenario: you are building a **developer support assistant** for *Team Workplace*, the team that owns **Roomie**, our internal meeting room booking service.

### Set up the environment
Before we can start, we have to set up the environment and several default values for model name, model parameters and prompts.

In [ ]:
# >>> SETUP <<<
# In Google Colab, this cell
#   1. clones the workshop repository,
#   2. installs its dependencies into the Python environment of this notebook (with uv),
#   3. makes the shared code (lib) importable.
# Locally (after `uv sync` in the repository), only step 3 is needed.
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/openknowledge/diy-copilot-bootcamp"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_DIR = Path("/content/diy-copilot")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run([sys.executable, str(REPO_DIR / "setup" / "install_dependencies.py")], check=True)
else:
    # The repository root is the first parent directory containing the shared code
    REPO_DIR = next(p for p in Path.cwd().resolve().parents if (p / "lib").is_dir() and (p / "pyproject.toml").exists())

# The repository is not an installed package, so we add it to the module search path
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print(f"Workshop repository: {REPO_DIR}")

In [2]:
import json
import time
from typing import Literal

from pydantic import BaseModel

from lib.completion import Chat, generate_completion
from lib.constants import KNOWLEDGE_BASE_PATH, TICKETS_PATH
from lib.environment import load_environment

# Load the API key and the default models. If you want, you can change the model.
default_models = load_environment()
MODEL_ID = default_models["generation_model"]
print(f"Using model: {MODEL_ID}")

Using model: gemini/gemini-3.1-flash-lite


### Definition of convenient functions

`Chat` (from `lib.completion`) handles multi-turn conversations – it keeps the message history automatically. `create_support_bot()` is a factory that creates a `Chat` with the tickets of Team Workplace's issue tracker injected into the system prompt.

In [3]:
# Set default values
DEFAULT_TEMPERATURE = 0.7
DEFAULT_MAX_TOKENS = 1000
DEFAULT_SYSTEM_PROMPT = "You are a friendly assistant"

# Load the issue tracker export to provide it as context to the model
tickets = json.loads(TICKETS_PATH.read_text(encoding="utf-8"))
tickets_content = json.dumps(tickets, indent=2)


def create_support_bot(system_prompt: str = DEFAULT_SYSTEM_PROMPT) -> Chat:
    """Creates a Chat instance with the issue tracker tickets injected into the system prompt."""
    full_system_prompt = f"{system_prompt}\n\nTickets of Team Workplace:\n{tickets_content}"
    return Chat(
        system_prompt=full_system_prompt,
        model_name=MODEL_ID,
        temperature=DEFAULT_TEMPERATURE,
        max_tokens=DEFAULT_MAX_TOKENS,
    )

In [4]:
def run_conversation(system_prompt: str, developer_prompts: list[str]):
    """Runs a multi-turn conversation with the support bot.

    Parameters
    ----------
    system_prompt : str
        The system prompt for the bot.
    developer_prompts : list[str]
        The list of developer messages to send in order.
    """
    bot = create_support_bot(system_prompt=system_prompt)
    for prompt in developer_prompts:
        print(f'Developer:\n{prompt}\n')
        answer = bot.send(prompt)
        print(f'Support bot:\n{answer}\n')

In [5]:
def watch_ticket(ticket_id: str):
    """Adds the current developer as a watcher to a ticket.

    Parameters
    ----------
    ticket_id : str
        The id of the ticket, e.g. "WORK-123".
    """
    # Simulate the call to the issue tracker API
    print(f"Adding you as a watcher to ticket {ticket_id}")
    for _ in range(5):
        print(".", end="", flush=True)
        time.sleep(0.5)
    print()
    if ticket_id == "WORK-412":
        print("Success: You are now watching 'WORK-412: 500 Internal Server Error when creating a booking without timezone'!")
        print("You completed this exercise successfully!")
    else:
        print(f"Error: {ticket_id} is not the ticket describing this problem.")

### Important notice: History
In this (and only this) lesson, we use a history feature so that the conversation can keep on going. Each exercise creates a fresh `Chat` instance, which starts with an empty history. If you mess up mid-exercise, just re-run the cell that creates the bot.

### Exercise 01: Flipped interaction
Your task is to create an assistant that gathers more information from the developer if the request is too vague.

Imagine the following situation:
A developer asks a senior colleague: *"My code doesn't work. Can you fix it?"*
The senior colleague can't help without more information and asks back: *"Sure – can you show me the code, the error message and what you expected to happen?"*

This reaction should now be done by the assistant. Your task is to provide a system prompt for this bot. The bot should:
* Ask the developer for more information if the provided information is not enough to help.
* Ask targeted questions instead of guessing a solution.

In [6]:
# Do not change the user prompt.
user_prompt = "My code for creating a booking doesn't work. Can you fix it?"

# TODO: Define the system prompt of the support bot using the "Flipped Interaction" pattern
system_prompt = """
You are a helpful senior developer supporting the developers who work with the Roomie service.
When a developer asks for help but provides incomplete information, do not guess a solution.
Instead, politely ask targeted follow-up questions, e.g. for the relevant code, the exact error message
or HTTP response, the expected behaviour, the request that was sent and the environment.
Once enough information is provided, help to solve the problem.
"""

run_conversation(system_prompt, [user_prompt])

Developer:
My code for creating a booking doesn't work. Can you fix it?

Support bot:
I would be happy to help you with that! To get started, could you please provide a bit more information so I can understand the issue?

Specifically, it would be helpful to see:

1.  **The code snippet** you are using to create the booking.
2.  **The exact request** you are sending (including the payload).
3.  **The error message** or the HTTP response (status code and body) you are receiving.
4.  **The environment** you are testing in (e.g., local, staging, production).

Once I have those details, I can look into why it isn't working for you. (Also, if you happen to be hitting the `500 Internal Server Error` described in **WORK-412**, please let me know!)



### Exercise 02: Basic workflow
Your task is to create a support bot that identifies a known problem from a developer's description and adds the developer as a watcher to the ticket on request.

Imagine the following situation: a developer from the mobile app team runs into an error.
The developer might ask in the team channel: *"When I create a booking from the mobile app, Roomie sometimes answers with a 500. Is this a known problem?"*
A member of Team Workplace knows the tickets and replies: *"Yes, that's WORK-412 – timestamps without timezone offset cause a TypeError. Should I add you as a watcher, so you get notified when it is fixed?"*

If the developer confirms (*"Yes, please!"*), the watcher is added. This is done by an issue tracker API that expects the ticket id, so the bot needs to finish the flow by replying with a line like `ticket = WORK-...`.

Your task is to provide a system prompt for this bot. The bot should:
* Respond to the developer by naming the ticket id, its title, its status and a short explanation of the cause.
* Ask whether the developer wants to watch the ticket.
* On confirmation, reply with the template `ticket = *id-of-the-ticket*`.

**Hint**: The creators of the bot (aka the teachers of this workshop) already provided the tickets to the bot. You do not need to worry about this.

In [7]:
# TODO: Define the respective system prompt.
system_prompt = (
    "You are a friendly support assistant of Team Workplace, the team that owns the Roomie service. "
    "Your task is to help developers who run into problems with Roomie. "
    "Use the tickets of Team Workplace to find out whether the problem is already known. "
    "If you find a matching ticket, provide the ticket id, the title, the status and a short explanation of the cause. "
    "In addition, ask the developer if you should add them as a watcher to the ticket. "
    "If the developer wants to watch the ticket, reply with the following template: 'ticket = *id-of-the-ticket*'"
)

In [8]:
run_conversation(system_prompt, [
    "When I create a booking from the mobile app, Roomie sometimes answers with a 500. Is this a known problem?",
    "Yes, please add me as a watcher.",
])

Developer:
When I create a booking from the mobile app, Roomie sometimes answers with a 500. Is this a known problem?

Support bot:
Hello! I'm sorry to hear you're running into 500 errors with the mobile app.

Yes, this is a known issue. It appears to be related to the format of the timestamps being sent.

*   **Ticket ID:** WORK-412
*   **Title:** 500 Internal Server Error when creating a booking without timezone
*   **Status:** Open
*   **Cause:** The API currently throws a 500 error instead of a 422 when `start` or `end` timestamps are sent without a timezone offset (e.g., `'2026-03-03T10:00:00'`). This causes a `TypeError` when the system tries to compare a naive datetime with a timezone-aware "now."
*   **Workaround:** As a temporary fix, please ensure your requests always include the timezone offset (e.g., `+01:00`) or use `'Z'` for UTC.

Would you like me to add you as a watcher to this ticket so you can stay updated on its progress?

Developer:
Yes, please add me as a watcher.


In [9]:
# TODO: Let's test if the bot identified the correct ticket.
# Copy the ticket id of the last bot response and assign it to the variable below.
# If you did it correctly, you should see a success message.
ticket_id = "WORK-412"
watch_ticket(ticket_id=ticket_id)

Adding you as a watcher to ticket WORK-412
.....
Success: You are now watching 'WORK-412: 500 Internal Server Error when creating a booking without timezone'!
You completed this exercise successfully!


### Exercise 03: Give choices
After identifying the ticket, the bot should give the developer another choice: in addition to watching the ticket, the bot can explain the **workaround** (if the ticket has one) and show a short code example that applies it. Update the system prompt in order to achieve this.

In [10]:
# TODO: Update the system prompt to include the option to explain the workaround instead of watching the ticket.
system_prompt = """
You are a friendly and knowledgeable support assistant of Team Workplace, the team that owns the Roomie
service, an internal FastAPI service for booking meeting rooms. Your primary task is to help developers
who run into problems with Roomie.

Instructions & Context:
1. Finding the ticket:
- Use the tickets of Team Workplace to find out whether the problem of the developer is already known.
- If the description of the developer is too vague, ask for more details (error code, endpoint, request).
2. Provide essential information:
- For the matching ticket, provide the following details:
  - Ticket id and title
  - Status
  - Short explanation of the cause
3. Offer two clear choices:
- Choice 1: You add the developer as a watcher to the ticket.
- Choice 2: You explain the workaround and show a short Python code example that applies it.
- Formulate these choices in a clear and engaging way, such as: Would you like me to add you as a watcher,
  or would you prefer that I show you the workaround?
- Only offer choice 2 if the ticket has a workaround.
4. Handling the developer's choice:
- If the developer wants the workaround, explain it and show the code example.
- If the developer wants to watch the ticket, respond with the following template: ticket = id-of-the-ticket

Examples:
Developer: My bookings for the Wannsee room get released although we are in the meeting.
Assistant: This is a known problem: WORK-398 "Bookings released although people are in the room" (status: in progress).
The door display of the room is offline, so nobody can check in and the no-show job releases the booking after 10 minutes.
Would you like me to add you as a watcher, or would you prefer that I show you the workaround?

Developer: Show me the workaround, please.
Assistant: Until the display is replaced, check in via Slack with `/room checkin <booking id>`.

Developer: Why are users getting two emails when a booking is released?
Assistant: This was WORK-405 "Users receive two release emails" (status: done). The scheduler was enabled on two replicas.
It is already fixed in the deployment configuration. Would you like me to add you as a watcher anyway?

Developer: Yes.
Assistant: ticket = WORK-405
"""

In [11]:
run_conversation(system_prompt, [
    "When I create a booking from the mobile app, Roomie sometimes answers with a 500. Is this a known problem?",
    "I can't wait for a fix. Show me the workaround!",
])

Developer:
When I create a booking from the mobile app, Roomie sometimes answers with a 500. Is this a known problem?

Support bot:
Yes, this is a known issue. It is tracked as **WORK-412 "500 Internal Server Error when creating a booking without timezone"** (status: open).

The error occurs because the API raises a `TypeError` when it tries to compare a naive datetime (sent without a timezone offset) with the timezone-aware "now" used by the system.

Would you like me to add you as a watcher to this ticket, or would you prefer that I show you the workaround?

Developer:
I can't wait for a fix. Show me the workaround!

Support bot:
To avoid the 500 error, you must ensure that all timestamps sent to the `POST /api/v2/bookings` endpoint include a timezone offset (e.g., `+01:00`) or the UTC indicator `Z`.

Here is a short Python example using `datetime` to ensure your timestamps are correctly formatted:

```python
from datetime import datetime, timezone

# Instead of using a naive datetim

In [12]:
# Test if the old process of watching a ticket still works
run_conversation(system_prompt, [
    "When I create a booking from the mobile app, Roomie sometimes answers with a 500. Is this a known problem?",
    "Please add me as a watcher, I want to know when it's fixed.",
])

Developer:
When I create a booking from the mobile app, Roomie sometimes answers with a 500. Is this a known problem?

Support bot:
Yes, this is a known issue. It is tracked as **WORK-412 "500 Internal Server Error when creating a booking without timezone"** (status: open).

The cause is that the service raises a `TypeError` when comparing a naive datetime (without a timezone offset) with the timezone-aware "now" on the server, causing the API to return a 500 error instead of a 422 validation error.

Would you like me to add you as a watcher to the ticket, or would you prefer that I show you the workaround?

Developer:
Please add me as a watcher, I want to know when it's fixed.

Support bot:
ticket = WORK-412



### (Optional) Exercise 04: Define the response format

In all previous exercises the bot replied with free-form text. That's fine for a human reader, but a coding assistant needs to *act* on the answer: show review comments next to the right line, apply an edit to a file, run a command. Any parsing code you write for free-form text will break as soon as the phrasing changes.

Most modern model providers solve this via a `response_format` parameter: you pass a **schema** describing the expected shape of the reply, and the model is constrained to produce JSON that matches it. A convenient way to define such a schema in Python is a **Pydantic model** – the same class serves as the schema for the model *and* as a validator for the reply.

In this exercise we build a small **code review assistant**. It reviews a code snippet against our company's **coding guidelines** and **logging requirements** and returns a list of findings. Each finding should contain:
* `line`: the line number in the snippet
* `rule`: the id of the violated rule (e.g. `PY-05`)
* `severity`: one of `"info"`, `"warning"`, `"error"`
* `comment`: what is wrong
* `suggestion`: the corrected code line

Your task:
1. Define a Pydantic model `ReviewFinding` with those fields and a model `CodeReview` containing a list of findings.
2. Pass `CodeReview` as `response_format=` to `generate_completion`.
3. Parse the JSON reply back into a `CodeReview` instance.

In [13]:
# The company guidelines the code has to follow
guidelines = "\n\n".join(
    (KNOWLEDGE_BASE_PATH / path).read_text(encoding="utf-8")
    for path in ["engineering/python-coding-guidelines.md", "security/secrets-and-logging.md"]
)

# A code snippet a developer wants to merge
code_to_review = '''\
from datetime import datetime, timedelta
from fastapi import HTTPException

def extend_booking(booking, minutes):
    if datetime.now() > booking.end:
        raise HTTPException(status_code=422, detail="Booking is already over")
    if (booking.end - booking.start).total_seconds() / 60 + minutes > 240:
        raise HTTPException(status_code=422, detail="Max 240 minutes")
    booking.end = booking.end + timedelta(minutes=minutes)
    print(f"Extended booking {booking.booking_id} for key {booking.api_key}")
    return booking
'''

# Show the snippet with line numbers - the model needs them to reference lines
numbered_code = "\n".join(f"{i:>2}: {line}" for i, line in enumerate(code_to_review.splitlines(), start=1))
print(numbered_code)

 1: from datetime import datetime, timedelta
 2: from fastapi import HTTPException
 3: 
 4: def extend_booking(booking, minutes):
 5:     if datetime.now() > booking.end:
 6:         raise HTTPException(status_code=422, detail="Booking is already over")
 7:     if (booking.end - booking.start).total_seconds() / 60 + minutes > 240:
 8:         raise HTTPException(status_code=422, detail="Max 240 minutes")
 9:     booking.end = booking.end + timedelta(minutes=minutes)
10:     print(f"Extended booking {booking.booking_id} for key {booking.api_key}")
11:     return booking


In [14]:
# TODO: Define Pydantic models describing the expected response shape.
class ReviewFinding(BaseModel):
    line: int
    rule: str
    severity: Literal["info", "warning", "error"]
    comment: str
    suggestion: str


class CodeReview(BaseModel):
    findings: list[ReviewFinding]


system_prompt = f"""
You are a strict code reviewer. Review the code of the developer against the company
guidelines below. Report every violation as a finding and reference the rule id.

Guidelines:
{guidelines}
"""

In [15]:
raw_reply = generate_completion(
    user_prompt=f"Please review this code:\n\n{numbered_code}",
    system_prompt=system_prompt,
    model_name=MODEL_ID,
    temperature=0.0,
    max_output_tokens=2000,
    response_format=CodeReview,
)

print(f"Raw model reply:\n{raw_reply}\n")

Raw model reply:
{
  "findings": [
    {
      "line": 2,
      "rule": "PY-06",
      "severity": "error",
      "comment": "Business logic must raise domain exceptions, not framework exceptions like HTTPException.",
      "suggestion": "Define a custom exception class and raise that instead."
    },
    {
      "line": 4,
      "rule": "PY-04",
      "severity": "error",
      "comment": "Public function is missing type hints.",
      "suggestion": "Add type hints for parameters and return type, e.g., def extend_booking(booking: Booking, minutes: int) -> Booking:"
    },
    {
      "line": 5,
      "rule": "PY-05",
      "severity": "error",
      "comment": "Direct call to datetime.now() is forbidden in business logic.",
      "suggestion": "Inject a clock callable to retrieve the current time."
    },
    {
      "line": 7,
      "rule": "PY-03",
      "severity": "error",
      "comment": "Magic number 240 detected.",
      "suggestion": "Move the maximum duration limit to config

In [16]:
# TODO: Parse the JSON reply back into a Pydantic instance for reliable access.
# Use the static class method `model_validate_json()` of the CodeReview class in order to validate the raw response!
review = CodeReview.model_validate_json(raw_reply)

# Now we can process the findings in code, e.g. render them like a review tool does
code_lines = code_to_review.splitlines()
for finding in sorted(review.findings, key=lambda f: f.line):
    code_line = code_lines[finding.line - 1].strip() if 0 < finding.line <= len(code_lines) else "?"
    print(f"Line {finding.line}: {code_line}")
    print(f"  [{finding.severity.upper()}] {finding.rule}: {finding.comment}")
    print(f"  Suggestion: {finding.suggestion}\n")

Line 2: from fastapi import HTTPException
  [ERROR] PY-06: Business logic must raise domain exceptions, not framework exceptions like HTTPException.
  Suggestion: Define a custom exception class and raise that instead.

Line 4: def extend_booking(booking, minutes):
  [ERROR] PY-04: Public function is missing type hints.
  Suggestion: Add type hints for parameters and return type, e.g., def extend_booking(booking: Booking, minutes: int) -> Booking:

Line 5: if datetime.now() > booking.end:
  [ERROR] PY-05: Direct call to datetime.now() is forbidden in business logic.
  Suggestion: Inject a clock callable to retrieve the current time.

Line 7: if (booking.end - booking.start).total_seconds() / 60 + minutes > 240:
  [ERROR] PY-03: Magic number 240 detected.
  Suggestion: Move the maximum duration limit to config.py and import it.

Line 10: print(f"Extended booking {booking.booking_id} for key {booking.api_key}")
  [ERROR] PY-10: Use of print() is forbidden in service code.
  Suggestion: U